In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.layers import GRU
from tensorflow.keras.optimizers import Adam
# Load the datasets
data_train = pd.read_csv('/content/sarcasm_tam_train.csv')
data_dev = pd.read_csv('/content/sarcasm_tam_dev.csv')
data_test = pd.read_csv('/content/sarcasm_tam_test_without_labels.csv')  # Assume this is your test dataset


# Combine the training and development sets
texts_combined = data_train['Text'].astype(str).values.tolist() + data_dev['Text'].astype(str).values.tolist()
labels_combined = np.concatenate([data_train['labels'].values, data_dev['labels'].values])

# Extract test texts
texts_test = data_test['Text'].astype(str).values.tolist()

# Tokenization parameters
max_words = 10000  # Maximum number of words to keep
max_len = 100      # Maximum sequence length

# Tokenize the text
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(texts_combined)  # Fit on combined texts
sequences_combined = tokenizer.texts_to_sequences(texts_combined)
sequences_test = tokenizer.texts_to_sequences(texts_test)
padded_sequences_combined = pad_sequences(sequences_combined, maxlen=max_len, padding='post')
padded_sequences_test = pad_sequences(sequences_test, maxlen=max_len, padding='post')

# Encode labels
label_encoder = LabelEncoder()
labels_encoded_combined = label_encoder.fit_transform(labels_combined)

# Convert labels to numpy arrays and ensure they are integers
labels_encoded_combined = np.array(labels_encoded_combined).astype(np.int32)

# Build the model
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=256, input_length=max_len))
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(32)))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))  # Add dense layer after LSTM to reduce dimensions
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))  # Output layer

# Compile the model
optimizer = Adam(learning_rate=0.0001)
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# Print model summary
model.summary()

# Train the model on the combined training set
model.fit(
    padded_sequences_combined,
    labels_encoded_combined,
    epochs=1,
    batch_size=32,
    validation_split=0.1  # Reserve 10% for validation during training
)

# Predict the labels for the test set
y_pred_test = (model.predict(padded_sequences_test) > 0.5).astype("int32")

# Map 0/1 predictions to "Non-sarcastic"/"Sarcastic"
label_mapping = {0: "Non-sarcastic", 1: "Sarcastic"}
predicted_labels = [label_mapping[label] for label in y_pred_test.flatten()]

# Create a DataFrame with predictions
predictions = pd.DataFrame({
    'Text': texts_test,
    'Predicted_Label': predicted_labels
})

# Save the predictions to a CSV file
predictions.to_csv('/content/Codespark_Tamil_run2.csv', index=False)

print("Predictions saved to Codespark_Tamil_run2.csv")

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_3 (Bidirectional)      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_4 (Bidirectional)      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_5 (Bidirectional)      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_6 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

1010/1010 ━━━━━━━━━━━━━━━━━━━━ 738s 720ms/step - accuracy: 0.7316 - loss: 0.5909 - val_accuracy: 0.7705 - val_loss: 0.4518
199/199 ━━━━━━━━━━━━━━━━━━━━ 41s 204ms/step
Predictions saved to Codespark_Tamil_run2.csv


In [ ]:
df = pd.read_csv('/content/Codespark_Tamil_run2.csv')

In [ ]:
df.head()

,Text,Predicted_Label
0,Kangana wow awesome yr ye lakdi sbae alh hai,Non-sarcastic
1,விழுப்புரம் வன்னிய கவுண்டர் சார்பாக வாழ்த்துக...,Non-sarcastic
2,திரௌபதி திரைப்படம் வெற்றி பெற வாணியர் சமுதாயம்...,Non-sarcastic
3,"இந்த திரைப்படம் வெற்றிபெற, ஆதி தமிழன் அதாவது இ...",Non-sarcastic
4,dai thala pera sonnalay summa tamil naday athi...,Sarcastic


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report
data_test_true = pd.read_csv('/content/sarcasm_tam_test.csv')
true_labels = data_test_true['labels'].values
y_pred_test_int = df['Predicted_Label'].values
print("Classification Report:\n")
print(classification_report(true_labels, y_pred_test_int, target_names=["Non-sarcastic", "Sarcastic"]))

Classification Report:

               precision    recall  f1-score   support

Non-sarcastic       0.85      0.84      0.84      4621
    Sarcastic       0.58      0.62      0.60      1717

     accuracy                           0.78      6338
    macro avg       0.72      0.73      0.72      6338
 weighted avg       0.78      0.78      0.78      6338

